#### Customer Segmentation profiling and customer Percona

##### Install packages

In [2]:
%pip install pandas pyarrow duckdb
%pip install --upgrade sqlalchemy
%pip install python-dotenv
%pip install "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# import all libraries
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

#### Setup Path

In [8]:
from pathlib import Path

ROOT = Path.cwd().parent

# Search for customer-related Parquet files
customer_files = [
    file for file in ROOT.rglob("*.parquet")
    if any(word in file.name.lower()
           for word in ["customer", "segment", "feature"])
]

print("CUSTOMER PARQUET FILES FOUND:\n")

if customer_files:
    for i, file in enumerate(customer_files, 1):
        print(f"{i}. {file}")
else:
    print("No customer-related Parquet files found.")

CUSTOMER PARQUET FILES FOUND:

1. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\Results\customer_features_engineered.parquet
2. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\Results\customer_segmentation_features.parquet
3. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\Results\customer_segmentation_ids.parquet
4. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\processed\veltrix_medium\customer_features.parquet
5. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\processed\veltrix_medium\customer_held_out_outcomes.parquet
6. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\processed\veltrix_medium\customer_kmeans_input.parquet
7. c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\processed\ve

In [9]:
# Search for ALL Parquet files in the project

all_parquet_files = list(ROOT.rglob("*.parquet"))

print("TOTAL PARQUET FILES:", len(all_parquet_files))

for i, file in enumerate(all_parquet_files, 1):
    print(f"{i}. {file.relative_to(ROOT)}")

TOTAL PARQUET FILES: 40
1. Results\customer_features_engineered.parquet
2. Results\customer_segmentation_features.parquet
3. Results\customer_segmentation_ids.parquet
4. data\processed\veltrix_medium\customer_features.parquet
5. data\processed\veltrix_medium\customer_held_out_outcomes.parquet
6. data\processed\veltrix_medium\customer_kmeans_input.parquet
7. data\processed\veltrix_medium\customer_kprototypes_input.parquet
8. data\raw\veltrix_medium\agents.parquet
9. data\raw\veltrix_medium\campaigns.parquet
10. data\raw\veltrix_medium\campaign_responses.parquet
11. data\raw\veltrix_medium\customers.parquet
12. data\raw\veltrix_medium\interactions.parquet
13. data\raw\veltrix_medium\leads.parquet
14. data\raw\veltrix_medium\locations.parquet
15. data\raw\veltrix_medium\payments.parquet
16. data\raw\veltrix_medium\properties.parquet
17. data\raw\veltrix_medium\site_visits.parquet
18. data\raw\veltrix_medium\transactions.parquet
19. data\raw\veltrix_medium\property_views\part-00000.parquet

In [10]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent

files_to_check = {
    "customer_features": ROOT / "data" / "processed" / "veltrix_medium" / "customer_features.parquet",
    "customer_features_engineered": ROOT / "Results" / "customer_features_engineered.parquet",
    "customer_segmentation_features": ROOT / "Results" / "customer_segmentation_features.parquet"
}

for name, file_path in files_to_check.items():

    print("\n" + "=" * 65)
    print(name.upper())
    print("=" * 65)

    df = pd.read_parquet(file_path)

    print("Shape:", df.shape)
    print("Customer ID available:", "customer_id" in df.columns)
    print("Columns:")
    print(df.columns.tolist())

    display(df.head(3))


CUSTOMER_FEATURES
Shape: (102000, 45)
Customer ID available: True
Columns:
['customer_id', 'email_sha256', 'phone_sha256', 'customer_type', 'gender', 'age_band', 'occupation_sector', 'country_of_residence', 'is_overseas', 'uk_region_of_residence', 'buyer_position', 'finance_type', 'primary_intent', 'stated_budget_gbp', 'stated_rent_budget_pcm_gbp', 'preferred_currency', 'first_contact_date', 'consent_marketing', 'created_at', 'total_sessions', 'active_session_days', 'avg_session_duration', 'total_session_duration', 'total_property_views', 'unique_properties_viewed', 'avg_views_per_property', 'avg_bedrooms_viewed', 'avg_property_price_viewed', 'avg_rent_viewed', 'property_types_explored', 'countries_explored', 'regions_explored', 'avg_viewed_property_price', 'min_viewed_property_price', 'max_viewed_property_price', 'avg_viewed_rent_pcm', 'min_viewed_rent_pcm', 'max_viewed_rent_pcm', 'avg_viewed_bedrooms', 'property_types_viewed', 'listing_purposes_viewed', 'areas_viewed', 'regions_view

,customer_id,email_sha256,phone_sha256,customer_type,gender,age_band,occupation_sector,country_of_residence,is_overseas,uk_region_of_residence,...,avg_viewed_rent_pcm,min_viewed_rent_pcm,max_viewed_rent_pcm,avg_viewed_bedrooms,property_types_viewed,listing_purposes_viewed,areas_viewed,regions_viewed,countries_viewed,locations_viewed
0,VLX-C00127559,cee942d4ef6870f680b2449f60b3752409a30deea7bb8f...,f25f93572ea10876a91e0c954f872681dfe44bc275af1f...,Individual,Male,18-24,None,United Kingdom,False,London,...,1990.26,700.0,4350.0,2.05,5.0,1.0,34.0,11.0,3.0,34.0
1,VLX-C00168848,1d312463c090da13d3f8e1ceeb5b9efcd8acfbf1c825d6...,34f46ff1ce0bc52cfe142176ae16aa8cf8bc86fbc9e9d2...,Individual,Male,25-34,None,United Kingdom,False,South East,...,NaN,NaN,NaN,1.64,4.0,1.0,14.0,8.0,3.0,14.0
2,VLX-C00155207,a89d26ac98e7b1b305848ee8acd6e519fbf302f2a05d12...,1abeda4811132a5aa955a909471d7462c27f6b8d123e84...,Individual,Female,65+,None,United Kingdom,False,East Midlands,...,NaN,NaN,NaN,3.13,5.0,1.0,9.0,6.0,2.0,9.0



CUSTOMER_FEATURES_ENGINEERED
Shape: (102000, 32)
Customer ID available: True
Columns:
['customer_id', 'email_sha256', 'phone_sha256', 'customer_type', 'gender', 'age_band', 'occupation_sector', 'country_of_residence', 'is_overseas', 'uk_region_of_residence', 'buyer_position', 'finance_type', 'primary_intent', 'stated_budget_gbp', 'stated_rent_budget_pcm_gbp', 'preferred_currency', 'first_contact_date', 'consent_marketing', 'created_at', 'total_sessions', 'avg_session_duration', 'total_session_duration', 'avg_views_per_property', 'avg_bedrooms_viewed', 'avg_property_price_viewed', 'avg_rent_viewed', 'property_types_explored', 'countries_explored', 'min_viewed_property_price', 'max_viewed_property_price', 'max_viewed_rent_pcm', 'night_session_share_x']


,customer_id,email_sha256,phone_sha256,customer_type,gender,age_band,occupation_sector,country_of_residence,is_overseas,uk_region_of_residence,...,avg_views_per_property,avg_bedrooms_viewed,avg_property_price_viewed,avg_rent_viewed,property_types_explored,countries_explored,min_viewed_property_price,max_viewed_property_price,max_viewed_rent_pcm,night_session_share_x
0,VLX-C00127559,cee942d4ef6870f680b2449f60b3752409a30deea7bb8f...,f25f93572ea10876a91e0c954f872681dfe44bc275af1f...,Individual,Male,18-24,Unknown,United Kingdom,False,London,...,1.85,2.05,NaN,1990.26,5.0,3.0,<NA>,<NA>,4350,0.2414
1,VLX-C00168848,1d312463c090da13d3f8e1ceeb5b9efcd8acfbf1c825d6...,34f46ff1ce0bc52cfe142176ae16aa8cf8bc86fbc9e9d2...,Individual,Male,25-34,Unknown,United Kingdom,False,South East,...,1.17,1.64,241554.05,NaN,4.0,3.0,105000,395000,<NA>,0.1667
2,VLX-C00155207,a89d26ac98e7b1b305848ee8acd6e519fbf302f2a05d12...,1abeda4811132a5aa955a909471d7462c27f6b8d123e84...,Individual,Female,65+,Unknown,United Kingdom,False,East Midlands,...,1.14,3.13,487343.75,NaN,5.0,2.0,250000,765000,<NA>,0.5000



CUSTOMER_SEGMENTATION_FEATURES
Shape: (102000, 15)
Customer ID available: False
Columns:
['stated_budget_gbp', 'stated_rent_budget_pcm_gbp', 'total_sessions', 'avg_session_duration', 'total_session_duration', 'avg_views_per_property', 'avg_bedrooms_viewed', 'avg_property_price_viewed', 'avg_rent_viewed', 'property_types_explored', 'countries_explored', 'min_viewed_property_price', 'max_viewed_property_price', 'max_viewed_rent_pcm', 'night_session_share_x']


,stated_budget_gbp,stated_rent_budget_pcm_gbp,total_sessions,avg_session_duration,total_session_duration,avg_views_per_property,avg_bedrooms_viewed,avg_property_price_viewed,avg_rent_viewed,property_types_explored,countries_explored,min_viewed_property_price,max_viewed_property_price,max_viewed_rent_pcm,night_session_share_x
0,-0.167595,2.060513,2.336679,0.0,0.0,2.130847,-0.591161,-0.215961,1.595245,0.583140,1.029165,-0.212558,-0.204833,3.407358,-0.285141
1,-0.625589,-0.080618,-0.471526,0.0,0.0,0.157393,-1.290998,-0.762688,-0.116001,-0.047422,1.029165,-0.793855,-0.972430,-0.075597,-0.626962
2,3.063808,-0.080618,-0.579534,0.0,0.0,0.070329,1.252310,0.980642,-0.116001,0.583140,-0.055275,1.079213,0.099309,-0.075597,0.898189


In [11]:
PROCESSED_DIR = ROOT / "data" / "processed" / "veltrix_medium"

customer_data = pd.read_parquet(
    PROCESSED_DIR / "customer_features.parquet"
)

print("Customer records:", len(customer_data))
print("Number of columns:", len(customer_data.columns))

display(customer_data.head())

Customer records: 102000
Number of columns: 45


,customer_id,email_sha256,phone_sha256,customer_type,gender,age_band,occupation_sector,country_of_residence,is_overseas,uk_region_of_residence,...,avg_viewed_rent_pcm,min_viewed_rent_pcm,max_viewed_rent_pcm,avg_viewed_bedrooms,property_types_viewed,listing_purposes_viewed,areas_viewed,regions_viewed,countries_viewed,locations_viewed
0,VLX-C00127559,cee942d4ef6870f680b2449f60b3752409a30deea7bb8f...,f25f93572ea10876a91e0c954f872681dfe44bc275af1f...,Individual,Male,18-24,None,United Kingdom,False,London,...,1990.26,700.0,4350.0,2.05,5.0,1.0,34.0,11.0,3.0,34.0
1,VLX-C00168848,1d312463c090da13d3f8e1ceeb5b9efcd8acfbf1c825d6...,34f46ff1ce0bc52cfe142176ae16aa8cf8bc86fbc9e9d2...,Individual,Male,25-34,None,United Kingdom,False,South East,...,NaN,NaN,NaN,1.64,4.0,1.0,14.0,8.0,3.0,14.0
2,VLX-C00155207,a89d26ac98e7b1b305848ee8acd6e519fbf302f2a05d12...,1abeda4811132a5aa955a909471d7462c27f6b8d123e84...,Individual,Female,65+,None,United Kingdom,False,East Midlands,...,NaN,NaN,NaN,3.13,5.0,1.0,9.0,6.0,2.0,9.0
3,VLX-C00247948,992ff440f07ec60b59ee61bb11c1849fcf992026d973fd...,9fb2f49fde40a903054b58ce406b556815de43bf82e7ec...,Individual,Female,45-54,Hospitality,United States,True,None,...,NaN,NaN,NaN,1.75,4.0,1.0,9.0,6.0,2.0,9.0
4,VLX-C00238325,e3f395363fc3a3ed6bd295c50cb64e4e26e6b8add80e74...,a659636e76f9e2f8a3e681de30fc5c37eb262b087f0fac...,Individual,Male,None,Technology,United Kingdom,False,South East,...,1346.88,825.0,1775.0,2.13,4.0,1.0,7.0,5.0,1.0,7.0


In [12]:
print("Duplicate customer IDs:",
      customer_data["customer_id"].duplicated().sum())

print("Missing customer IDs:",
      customer_data["customer_id"].isna().sum())

print("Unique customers:",
      customer_data["customer_id"].nunique())

Duplicate customer IDs: 0
Missing customer IDs: 0
Unique customers: 102000


In [ ]:
######=======> Correct the missing-file error
customer_data = pd.read_parquet(
    PROCESSED_DIR / "customer_features.parquet"
)

print("Customer records:", len(customer_data))
print("Customer features:", len(customer_data.columns))

display(customer_data.head())

Customer records: 102000
Customer features: 45


,customer_id,email_sha256,phone_sha256,customer_type,gender,age_band,occupation_sector,country_of_residence,is_overseas,uk_region_of_residence,...,avg_viewed_rent_pcm,min_viewed_rent_pcm,max_viewed_rent_pcm,avg_viewed_bedrooms,property_types_viewed,listing_purposes_viewed,areas_viewed,regions_viewed,countries_viewed,locations_viewed
0,VLX-C00127559,cee942d4ef6870f680b2449f60b3752409a30deea7bb8f...,f25f93572ea10876a91e0c954f872681dfe44bc275af1f...,Individual,Male,18-24,None,United Kingdom,False,London,...,1990.26,700.0,4350.0,2.05,5.0,1.0,34.0,11.0,3.0,34.0
1,VLX-C00168848,1d312463c090da13d3f8e1ceeb5b9efcd8acfbf1c825d6...,34f46ff1ce0bc52cfe142176ae16aa8cf8bc86fbc9e9d2...,Individual,Male,25-34,None,United Kingdom,False,South East,...,NaN,NaN,NaN,1.64,4.0,1.0,14.0,8.0,3.0,14.0
2,VLX-C00155207,a89d26ac98e7b1b305848ee8acd6e519fbf302f2a05d12...,1abeda4811132a5aa955a909471d7462c27f6b8d123e84...,Individual,Female,65+,None,United Kingdom,False,East Midlands,...,NaN,NaN,NaN,3.13,5.0,1.0,9.0,6.0,2.0,9.0
3,VLX-C00247948,992ff440f07ec60b59ee61bb11c1849fcf992026d973fd...,9fb2f49fde40a903054b58ce406b556815de43bf82e7ec...,Individual,Female,45-54,Hospitality,United States,True,None,...,NaN,NaN,NaN,1.75,4.0,1.0,9.0,6.0,2.0,9.0
4,VLX-C00238325,e3f395363fc3a3ed6bd295c50cb64e4e26e6b8add80e74...,a659636e76f9e2f8a3e681de30fc5c37eb262b087f0fac...,Individual,Male,None,Technology,United Kingdom,False,South East,...,1346.88,825.0,1775.0,2.13,4.0,1.0,7.0,5.0,1.0,7.0


In [ ]:
#####======> Rebuild the segmentation dataset with customer IDs

ROOT = Path.cwd().parent

PROCESSED_DIR = ROOT / "data" / "processed" / "veltrix_medium"
RESULTS_DIR = ROOT / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load complete customer features
customer_data = pd.read_parquet(
    PROCESSED_DIR / "customer_features.parquet"
)

# Load existing scaled segmentation features
segmentation_features = pd.read_parquet(
    RESULTS_DIR / "customer_segmentation_features.parquet"
)

# Check row alignment before attaching customer IDs
assert len(customer_data) == len(segmentation_features), (
    "Row counts do not match. Do not attach IDs until alignment is verified."
)

# Rebuild customer-level segmentation base
customer_segmentation_base = pd.concat(
    [
        customer_data[["customer_id"]].reset_index(drop=True),
        segmentation_features.reset_index(drop=True)
    ],
    axis=1
)

# Check results
print("Shape:", customer_segmentation_base.shape)
print("Unique customers:", customer_segmentation_base["customer_id"].nunique())
print("Missing IDs:", customer_segmentation_base["customer_id"].isna().sum())

display(customer_segmentation_base.head())

# Save corrected file
customer_segmentation_base.to_parquet(
    PROCESSED_DIR / "customer_segmentation_base.parquet",
    index=False
)

print("Saved:", PROCESSED_DIR / "customer_segmentation_base.parquet")

Shape: (102000, 16)
Unique customers: 102000
Missing IDs: 0


,customer_id,stated_budget_gbp,stated_rent_budget_pcm_gbp,total_sessions,avg_session_duration,total_session_duration,avg_views_per_property,avg_bedrooms_viewed,avg_property_price_viewed,avg_rent_viewed,property_types_explored,countries_explored,min_viewed_property_price,max_viewed_property_price,max_viewed_rent_pcm,night_session_share_x
0,VLX-C00127559,-0.167595,2.060513,2.336679,0.0,0.0,2.130847,-0.591161,-0.215961,1.595245,0.583140,1.029165,-0.212558,-0.204833,3.407358,-0.285141
1,VLX-C00168848,-0.625589,-0.080618,-0.471526,0.0,0.0,0.157393,-1.290998,-0.762688,-0.116001,-0.047422,1.029165,-0.793855,-0.972430,-0.075597,-0.626962
2,VLX-C00155207,3.063808,-0.080618,-0.579534,0.0,0.0,0.070329,1.252310,0.980642,-0.116001,0.583140,-0.055275,1.079213,0.099309,-0.075597,0.898189
3,VLX-C00247948,-0.803698,-0.080618,-0.525530,0.0,0.0,-0.335970,-1.103237,-1.222328,-0.116001,-0.047422,-0.055275,-0.664677,-1.117259,-0.075597,0.440598
4,VLX-C00238325,-0.167595,-0.452989,-0.687542,0.0,0.0,-0.335970,-0.454608,-0.215961,-0.587185,-0.047422,-1.139714,-0.212558,-0.204833,-2.111786,-1.389766


Saved: c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\processed\veltrix_medium\customer_segmentation_base.parquet


#### Set up Path

In [17]:
# =====================================================
# 1. PROJECT DIRECTORIES
# =====================================================

# Project root: one level above notebooks
ROOT = Path.cwd().parent

PROCESSED_DIR = ROOT / "data" / "processed" / "veltrix_medium"
RESULTS_DIR = ROOT / "Results"

PROFILE_DIR = RESULTS_DIR / "reports" / "segment_profiles"
CHART_DIR = PROFILE_DIR / "charts"
PERSONA_DIR = PROFILE_DIR / "personas"

# Create output folders
for folder in [PROFILE_DIR, CHART_DIR, PERSONA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Processed directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)


# =====================================================
# 2. FIND AND CONNECT TO DUCKDB
# =====================================================

db_files = list(ROOT.rglob("*.duckdb"))

print("\nDuckDB database files found:")

for file in db_files:
    print(file)

if not db_files:
    raise FileNotFoundError(
        f"No DuckDB database found under: {ROOT}"
    )

# If multiple database files exist, select the correct one
DB_PATH = db_files[0]

print("\nConnecting to:", DB_PATH)

con = duckdb.connect(
    database=str(DB_PATH),
    read_only=False
)

print("DuckDB connection established successfully!")


# =====================================================
# 3. CHECK AVAILABLE TABLES
# =====================================================

print("\nAvailable tables and views:")

display(
    con.sql("SHOW TABLES").df()
)

print("\nAvailable schemas:")

display(
    con.sql("""
        SELECT schema_name
        FROM information_schema.schemata
        ORDER BY schema_name
    """).df()
)


# =====================================================
# 4. LOAD CUSTOMER SEGMENTATION DATASET
# =====================================================

CUSTOMER_FILE = (
    PROCESSED_DIR / "customer_segmentation_base.parquet"
)

if not CUSTOMER_FILE.exists():
    raise FileNotFoundError(
        f"Customer segmentation dataset not found: {CUSTOMER_FILE}"
    )

customer_data = pd.read_parquet(CUSTOMER_FILE)

print("\nCustomer segmentation dataset loaded successfully!")

print("Customer records:", f"{len(customer_data):,}")
print("Number of features:", len(customer_data.columns))

print("\nCustomer columns:")

print(customer_data.columns.tolist())


# =====================================================
# 5. INITIAL DATA VALIDATION
# =====================================================

print("\nDataset dimensions:", customer_data.shape)

print(
    "Duplicate customer IDs:",
    customer_data["customer_id"].duplicated().sum()
    if "customer_id" in customer_data.columns
    else "customer_id column not found"
)

print("\nMissing values:")

display(
    customer_data.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
    .query("missing_values > 0")
    .head(20)
)

Project root: c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F
Processed directory: c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\processed\veltrix_medium
Results directory: c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\Results

DuckDB database files found:
c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\veltrix_customer_segmentation.duckdb
c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\data\veltrix.duckdb

Connecting to: c:\Users\User\Documents\AMDARI_IMS\veltrix_customer_segmentation\veltrix-segmentation_F\veltrix_customer_segmentation.duckdb
DuckDB connection established successfully!

Available tables and views:


,name



Available schemas:


,schema_name
0,information_schema
1,main
2,main
3,main
4,pg_catalog



Customer segmentation dataset loaded successfully!
Customer records: 102,000
Number of features: 16

Customer columns:
['customer_id', 'stated_budget_gbp', 'stated_rent_budget_pcm_gbp', 'total_sessions', 'avg_session_duration', 'total_session_duration', 'avg_views_per_property', 'avg_bedrooms_viewed', 'avg_property_price_viewed', 'avg_rent_viewed', 'property_types_explored', 'countries_explored', 'min_viewed_property_price', 'max_viewed_property_price', 'max_viewed_rent_pcm', 'night_session_share_x']

Dataset dimensions: (102000, 16)
Duplicate customer IDs: 0

Missing values:


,missing_values


In [24]:
# ============================================================
# REGISTER ALL VELTRIX PARQUET DATASETS IN DUCKDB
# ============================================================

from pathlib import Path

ROOT = Path.cwd().parent

RAW_DIR = ROOT / "data" / "raw" / "veltrix_medium"

# Check source folder
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {RAW_DIR}")

# Find all Parquet files
parquet_files = list(RAW_DIR.rglob("*.parquet"))

# Exclude restricted answer key
parquet_files = [
    file for file in parquet_files
    if "_restricted" not in file.parts
]

# Identify unique dataset names
table_files = {}

for file in parquet_files:

    if file.parent == RAW_DIR:
        table_name = file.stem
    else:
        table_name = file.parent.name

    table_files[table_name] = file.parent if file.parent != RAW_DIR else file

# Register each dataset
registered_tables = []
failed_tables = []

for table_name, file_path in table_files.items():

    try:
        parquet_path = file_path.as_posix().replace("'", "''")

        con.execute(f"""
            CREATE OR REPLACE VIEW "{table_name}" AS
            SELECT *
            FROM read_parquet('{parquet_path}')
        """)

        registered_tables.append(table_name)

        print(f"Registered successfully: {table_name}")

    except Exception as e:

        failed_tables.append({
            "table": table_name,
            "error": str(e)
        })

        print(f"Registration failed: {table_name}")
        print(e)


# Registration summary
print("\n" + "=" * 55)
print("DUCKDB REGISTRATION SUMMARY")
print("=" * 55)

print("Successfully registered:", len(registered_tables))
print("Failed:", len(failed_tables))

print("\nRegistered tables:")
print(registered_tables)

if failed_tables:
    print("\nFailed tables:")
    display(pd.DataFrame(failed_tables))


# Display available DuckDB views
print("\nAVAILABLE DUCKDB TABLES")

display(
    con.sql("SHOW TABLES").df()
)

Registered successfully: agents
Registered successfully: campaigns
Registered successfully: campaign_responses
Registered successfully: customers
Registered successfully: interactions
Registered successfully: leads
Registered successfully: locations
Registered successfully: payments
Registered successfully: properties
Registered successfully: site_visits
Registered successfully: transactions
Registered successfully: property_views
Registered successfully: web_sessions

DUCKDB REGISTRATION SUMMARY
Successfully registered: 13
Failed: 0

Registered tables:
['agents', 'campaigns', 'campaign_responses', 'customers', 'interactions', 'leads', 'locations', 'payments', 'properties', 'site_visits', 'transactions', 'property_views', 'web_sessions']

AVAILABLE DUCKDB TABLES


,name
0,agents
1,campaign_responses
2,campaigns
3,customers
4,interactions
5,leads
6,locations
7,payments
8,properties
9,property_views


In [25]:
#####========> inspect actual lead and trasaction
# Inspect lead stages
print("LEAD STAGES")

display(
    con.sql("""
        SELECT
            lead_stage,
            COUNT(*) AS lead_count
        FROM leads
        GROUP BY lead_stage
        ORDER BY lead_count DESC
    """).df()
)

# Inspect lead conversion values
print("LEAD CONVERSION VALUES")

display(
    con.sql("""
        SELECT
            converted,
            COUNT(*) AS lead_count
        FROM leads
        GROUP BY converted
    """).df()
)

# Inspect transaction types
print("TRANSACTION TYPES")

display(
    con.sql("""
        SELECT
            transaction_type,
            COUNT(*) AS transaction_count
        FROM transactions
        GROUP BY transaction_type
        ORDER BY transaction_count DESC
    """).df()
)

# Inspect transaction completion dates
print("TRANSACTION COMPLETION DATA")

display(
    con.sql("""
        SELECT
            COUNT(*) AS total_transactions,
            COUNT(completion_or_move_in_date) AS completed_date_count,
            COUNT(contract_date) AS contract_date_count
        FROM transactions
    """).df()
)

# Inspect property listing purposes
print("PROPERTY LISTING PURPOSES")

display(
    con.sql("""
        SELECT
            listing_purpose,
            COUNT(*) AS property_count
        FROM properties
        GROUP BY listing_purpose
        ORDER BY property_count DESC
    """).df()
)

LEAD STAGES


,lead_stage,lead_count
0,Lost,74759
1,Won,16665
2,Viewing,1119
3,Qualified,1007
4,Contacted,874
5,Offer,853
6,New,527


LEAD CONVERSION VALUES


,converted,lead_count
0,True,16665
1,False,79139


TRANSACTION TYPES


,transaction_type,transaction_count
0,Resale purchase,9063
1,Assured shorthold tenancy,3327
2,New-build purchase,1866
3,Company let,1311
4,Off-plan purchase,1098


TRANSACTION COMPLETION DATA


,total_transactions,completed_date_count,contract_date_count
0,16665,16665,16665


PROPERTY LISTING PURPOSES


,listing_purpose,property_count
0,Sale,12550
1,Rent,4117


#### Build customer-level behavioural and outcome features
This cell summarises leads, transactions, sessions, property views, site visits and agent interactions for each customer.

In [26]:
# ---------------------------------------------
# A. LEAD FEATURES
# ---------------------------------------------

lead_features = con.sql("""
    SELECT
        customer_id,

        COUNT(DISTINCT lead_id) AS total_leads,

        COUNT(DISTINCT CASE
            WHEN LOWER(CAST(converted AS VARCHAR))
                 IN ('true', '1', 'yes', 'y')
            THEN lead_id
        END) AS converted_leads,

        COUNT(DISTINCT CASE
            WHEN LOWER(CAST(converted AS VARCHAR))
                 NOT IN ('true', '1', 'yes', 'y')
                 OR converted IS NULL
            THEN lead_id
        END) AS open_or_unconverted_leads,

        MIN(TRY_CAST(created_date AS TIMESTAMP))
            AS first_lead_date,

        MAX(TRY_CAST(created_date AS TIMESTAMP))
            AS latest_lead_date,

        MAX(TRY_CAST(closed_date AS TIMESTAMP))
            AS latest_closed_date

    FROM leads
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
""").df()


# ---------------------------------------------
# B. TRANSACTION FEATURES
# ---------------------------------------------

transaction_features = con.sql("""
    SELECT
        customer_id,

        COUNT(DISTINCT transaction_id)
            AS total_transactions,

        COUNT(DISTINCT CASE
            WHEN completion_or_move_in_date IS NOT NULL
            THEN transaction_id
        END) AS completed_transactions,

        SUM(
            CASE
                WHEN completion_or_move_in_date IS NOT NULL
                     AND LOWER(COALESCE(transaction_type, ''))
                         NOT LIKE '%rent%'
                THEN COALESCE(agreed_price_gbp, 0)
                ELSE 0
            END
        ) AS completed_sale_value_gbp,

        SUM(
            CASE
                WHEN completion_or_move_in_date IS NOT NULL
                     AND LOWER(COALESCE(transaction_type, ''))
                         LIKE '%rent%'
                THEN COALESCE(agreed_rent_pcm_gbp, 0)
                ELSE 0
            END
        ) AS completed_rental_value_pcm_gbp,

        MAX(
            TRY_CAST(completion_or_move_in_date AS TIMESTAMP)
        ) AS latest_completion_date

    FROM transactions
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
""").df()


# ---------------------------------------------
# C. WEB SESSION FEATURES
# ---------------------------------------------

session_features = con.sql("""
    SELECT
        customer_id,

        COUNT(DISTINCT session_id) AS total_sessions,

        COUNT(DISTINCT CAST(session_start AS DATE))
            AS active_session_days,

        AVG(TRY_CAST(duration_sec AS DOUBLE))
            AS avg_session_duration_sec,

        SUM(TRY_CAST(duration_sec AS DOUBLE))
            AS total_session_duration_sec,

        MAX(TRY_CAST(session_start AS TIMESTAMP))
            AS latest_session_date

    FROM web_sessions
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
""").df()


# ---------------------------------------------
# D. PROPERTY VIEW FEATURES
# ---------------------------------------------

property_view_features = con.sql("""
    SELECT
        ws.customer_id,

        COUNT(DISTINCT pv.view_id)
            AS total_property_views,

        COUNT(DISTINCT pv.property_id)
            AS unique_properties_viewed,

        COUNT(DISTINCT p.property_type)
            AS property_types_viewed,

        MODE(p.property_type)
            AS most_viewed_property_type,

        MODE(p.listing_purpose)
            AS most_viewed_listing_purpose,

        MEDIAN(p.asking_price_gbp)
            AS median_viewed_sale_price_gbp,

        MEDIAN(p.rent_pcm_gbp)
            AS median_viewed_rent_pcm_gbp,

        MEDIAN(p.bedrooms)
            AS median_viewed_bedrooms,

        MAX(TRY_CAST(pv.event_time AS TIMESTAMP))
            AS latest_property_view_date

    FROM property_views pv

    INNER JOIN web_sessions ws
        ON pv.session_id = ws.session_id

    LEFT JOIN properties p
        ON pv.property_id = p.property_id

    WHERE ws.customer_id IS NOT NULL

    GROUP BY ws.customer_id
""").df()


# ---------------------------------------------
# E. SITE VISIT FEATURES
# ---------------------------------------------

site_visit_features = con.sql("""
    SELECT
        l.customer_id,

        COUNT(DISTINCT sv.visit_id)
            AS total_site_visits,

        COUNT(DISTINCT CASE
            WHEN LOWER(COALESCE(sv.outcome, ''))
                 LIKE '%success%'
            THEN sv.visit_id
        END) AS successful_site_visits,

        MAX(TRY_CAST(sv.visit_date AS TIMESTAMP))
            AS latest_site_visit_date

    FROM site_visits sv

    INNER JOIN leads l
        ON sv.lead_id = l.lead_id

    WHERE l.customer_id IS NOT NULL

    GROUP BY l.customer_id
""").df()


# ---------------------------------------------
# F. AGENT INTERACTION FEATURES
# ---------------------------------------------

interaction_features = con.sql("""
    SELECT
        l.customer_id,

        COUNT(DISTINCT i.interaction_id)
            AS total_interactions,

        MEDIAN(
            TRY_CAST(i.response_time_min AS DOUBLE)
        ) AS median_recorded_response_min,

        COUNT(DISTINCT CASE
            WHEN LOWER(COALESCE(i.direction, '')) = 'outbound'
            THEN i.interaction_id
        END) AS outbound_interactions,

        COUNT(DISTINCT CASE
            WHEN LOWER(COALESCE(i.direction, '')) = 'inbound'
            THEN i.interaction_id
        END) AS inbound_interactions,

        MAX(TRY_CAST(i.interaction_time AS TIMESTAMP))
            AS latest_interaction_date

    FROM interactions i

    INNER JOIN leads l
        ON i.lead_id = l.lead_id

    WHERE l.customer_id IS NOT NULL

    GROUP BY l.customer_id
""").df()


print("Lead features:", lead_features.shape)
print("Transaction features:", transaction_features.shape)
print("Session features:", session_features.shape)
print("Property view features:", property_view_features.shape)
print("Site visit features:", site_visit_features.shape)
print("Interaction features:", interaction_features.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lead features: (61030, 7)
Transaction features: (14985, 6)
Session features: (96368, 6)
Property view features: (96096, 10)
Site visit features: (55132, 4)
Interaction features: (61030, 6)


#### Combine the customer behaviour features

In [27]:
# Start with the original customer-level base
customer_profile = customer_data.copy()

# Check customer key
assert "customer_id" in customer_profile.columns

assert customer_profile["customer_id"].is_unique, (
    "The customer base contains duplicate customer IDs."
)

# Aggregated feature tables
feature_tables = [
    lead_features,
    transaction_features,
    session_features,
    property_view_features,
    site_visit_features,
    interaction_features
]

# Merge each table
for feature_table in feature_tables:

    customer_profile = customer_profile.merge(
        feature_table,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )

# Fill missing count columns with zero
count_columns = [
    "total_leads",
    "converted_leads",
    "open_or_unconverted_leads",
    "total_transactions",
    "completed_transactions",
    "total_sessions",
    "active_session_days",
    "total_property_views",
    "unique_properties_viewed",
    "property_types_viewed",
    "total_site_visits",
    "successful_site_visits",
    "total_interactions",
    "outbound_interactions",
    "inbound_interactions"
]

for column in count_columns:

    if column in customer_profile.columns:
        customer_profile[column] = (
            pd.to_numeric(
                customer_profile[column],
                errors="coerce"
            )
            .fillna(0)
        )

# Calculate customer-level lead conversion rate
customer_profile["lead_conversion_rate"] = np.where(
    customer_profile["total_leads"] > 0,
    customer_profile["converted_leads"]
    / customer_profile["total_leads"],
    np.nan
)

# Customer-level transaction rate will be calculated
# in the segment report, using completed transactions.

print("Combined customer profile:", customer_profile.shape)

display(customer_profile.head())

Combined customer profile: (102000, 50)


,customer_id,stated_budget_gbp,stated_rent_budget_pcm_gbp,total_sessions_x,avg_session_duration,total_session_duration,avg_views_per_property,avg_bedrooms_viewed,avg_property_price_viewed,avg_rent_viewed,...,latest_property_view_date,total_site_visits,successful_site_visits,latest_site_visit_date,total_interactions,median_recorded_response_min,outbound_interactions,inbound_interactions,latest_interaction_date,lead_conversion_rate
0,VLX-C00127559,-0.167595,2.060513,2.336679,0.0,0.0,2.130847,-0.591161,-0.215961,1.595245,...,2023-03-17 21:20:19,0.0,0.0,NaT,0.0,NaN,0.0,0.0,NaT,NaN
1,VLX-C00168848,-0.625589,-0.080618,-0.471526,0.0,0.0,0.157393,-1.290998,-0.762688,-0.116001,...,2023-04-11 22:18:26,0.0,0.0,NaT,0.0,NaN,0.0,0.0,NaT,NaN
2,VLX-C00155207,3.063808,-0.080618,-0.579534,0.0,0.0,0.070329,1.252310,0.980642,-0.116001,...,2023-06-11 22:37:01,1.0,0.0,2023-03-04,5.0,45.5,3.0,2.0,2023-04-09 09:31:53,0.0
3,VLX-C00247948,-0.803698,-0.080618,-0.525530,0.0,0.0,-0.335970,-1.103237,-1.222328,-0.116001,...,2024-04-22 03:47:34,0.0,0.0,NaT,0.0,NaN,0.0,0.0,NaT,NaN
4,VLX-C00238325,-0.167595,-0.452989,-0.687542,0.0,0.0,-0.335970,-0.454608,-0.215961,-0.587185,...,2023-01-05 15:27:41,0.0,0.0,NaT,0.0,NaN,0.0,0.0,NaT,NaN


In [29]:
# Inspect columns containing session-related features

session_columns = [
    col for col in customer_profile.columns
    if "session" in col.lower()
]

print("SESSION COLUMNS:")
print(session_columns)

# Check whether duplicate session features exist
for col in ["total_sessions_x", "total_sessions_y"]:
    if col in customer_profile.columns:
        print(f"\n{col}")
        print(customer_profile[col].describe())

SESSION COLUMNS:
['total_sessions_x', 'avg_session_duration', 'total_session_duration', 'night_session_share_x', 'total_sessions_y', 'active_session_days', 'avg_session_duration_sec', 'total_session_duration_sec', 'latest_session_date']

total_sessions_x
count    1.020000e+05
mean     3.120815e-17
std      1.000005e+00
min     -7.955494e-01
25%     -5.795337e-01
50%     -3.095140e-01
75%      2.305253e-01
max      2.285817e+01
Name: total_sessions_x, dtype: float64

total_sessions_y
count    96368.000000
mean        15.592261
std         18.695037
min          1.000000
25%          4.000000
50%         10.000000
75%         19.000000
max        438.000000
Name: total_sessions_y, dtype: float64


In [30]:
# ============================================================
# FIX DUPLICATE SESSION FEATURE COLUMNS
# ============================================================

if "total_sessions" not in customer_profile.columns:

    if (
        "total_sessions_x" in customer_profile.columns
        and "total_sessions_y" in customer_profile.columns
    ):
        customer_profile["total_sessions"] = (
            customer_profile["total_sessions_x"]
            .combine_first(customer_profile["total_sessions_y"])
        )

    elif "total_sessions_x" in customer_profile.columns:
        customer_profile["total_sessions"] = (
            customer_profile["total_sessions_x"]
        )

    elif "total_sessions_y" in customer_profile.columns:
        customer_profile["total_sessions"] = (
            customer_profile["total_sessions_y"]
        )

# Verify
print("total_sessions restored:",
      "total_sessions" in customer_profile.columns)

print(
    customer_profile["total_sessions"].describe()
)

total_sessions restored: True
count    1.020000e+05
mean     3.120815e-17
std      1.000005e+00
min     -7.955494e-01
25%     -5.795337e-01
50%     -3.095140e-01
75%      2.305253e-01
max      2.285817e+01
Name: total_sessions, dtype: float64


#### Create the eight behavioural persona flags
The definitions below deliberately allow overlap. For example, a customer can be an overseas buyer, an investment-focused customer and a high-intent buyer at the same time.

In [32]:
# ---------------------------------------------
# A. ADD LEAD STAGE INFORMATION
# ---------------------------------------------

lead_stage_features = con.sql("""
    SELECT
        customer_id,

        STRING_AGG(
            DISTINCT LOWER(COALESCE(lead_stage, '')),
            ' '
        ) AS customer_lead_stages

    FROM leads
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
""").df()

customer_profile = customer_profile.merge(
    lead_stage_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# ---------------------------------------------
# B. STANDARDISE TEXT FIELDS
# ---------------------------------------------

text_columns = [
    "primary_intent",
    "finance_type",
    "buyer_position",
    "customer_type",
    "country_of_residence",
    "most_viewed_property_type",
    "most_viewed_listing_purpose",
    "customer_lead_stages"
]

for column in text_columns:

    if column in customer_profile.columns:
        customer_profile[column] = (
            customer_profile[column]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )

# ---------------------------------------------
# C. CALCULATE ACTIVITY RECENCY
# ---------------------------------------------

activity_date_columns = [
    "latest_lead_date",
    "latest_session_date",
    "latest_property_view_date",
    "latest_site_visit_date",
    "latest_interaction_date"
]

available_date_columns = [
    col for col in activity_date_columns
    if col in customer_profile.columns
]

for column in available_date_columns:
    customer_profile[column] = pd.to_datetime(
        customer_profile[column],
        errors="coerce",
        utc=True
    )

# Latest known customer activity
customer_profile["last_observed_activity"] = (
    customer_profile[available_date_columns].max(axis=1)
)

# Days since latest observed activity
reference_date = pd.Timestamp.now(tz="UTC")

customer_profile["days_since_observed_activity"] = (
    reference_date - customer_profile["last_observed_activity"]
).dt.total_seconds() / 86400

# ---------------------------------------------
# D. DATA-DRIVEN ENGAGEMENT THRESHOLDS
# ---------------------------------------------

views_threshold = customer_profile[
    "total_property_views"
].quantile(0.75)

sessions_threshold = customer_profile[
    "total_sessions"
].quantile(0.75)

unique_properties_threshold = customer_profile[
    "unique_properties_viewed"
].quantile(0.75)

print("75th percentile property views:", views_threshold)
print("75th percentile sessions:", sessions_threshold)
print("75th percentile unique properties:", unique_properties_threshold)

# ---------------------------------------------
# E. PREPARE CUSTOMER ATTRIBUTES
# ---------------------------------------------

def as_boolean(series):
    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .isin(["true", "1", "yes", "y"])
    )

if "is_overseas" in customer_profile.columns:
    overseas_flag = as_boolean(
        customer_profile["is_overseas"]
    )
else:
    overseas_flag = pd.Series(
        False,
        index=customer_profile.index
    )

primary_intent = customer_profile.get(
    "primary_intent",
    pd.Series("", index=customer_profile.index)
)

finance_type = customer_profile.get(
    "finance_type",
    pd.Series("", index=customer_profile.index)
)

buyer_position = customer_profile.get(
    "buyer_position",
    pd.Series("", index=customer_profile.index)
)

listing_purpose = customer_profile.get(
    "most_viewed_listing_purpose",
    pd.Series("", index=customer_profile.index)
)

lead_stages = customer_profile.get(
    "customer_lead_stages",
    pd.Series("", index=customer_profile.index)
)

# ---------------------------------------------
# F. DEFINE THE EIGHT CUSTOMER PERSONAS
# ---------------------------------------------

# 1. Repeat Buyers
customer_profile["persona_repeat_buyer"] = (
    customer_profile["completed_transactions"] >= 2
)

# 2. High-Intent Buyers
high_intent_keywords = (
    r"buy|purchase|buyer|home ownership|owner occupier"
)

advanced_stage_keywords = (
    r"viewing|offer|negotiat|reserved|reservation|"
    r"contract|completion|exchange"
)

has_purchase_intent = primary_intent.str.contains(
    high_intent_keywords,
    regex=True,
    na=False
)

has_advanced_stage = lead_stages.str.contains(
    advanced_stage_keywords,
    regex=True,
    na=False
)

customer_profile["persona_high_intent_buyer"] = (
    (customer_profile["total_leads"] > 0)
    & (
        has_purchase_intent
        | has_advanced_stage
        | (customer_profile["total_site_visits"] > 0)
    )
    & ~customer_profile["persona_repeat_buyer"]
)

# 3. Investment-Focused Customers
investment_keywords = (
    r"invest|buy.to.let|landlord|capital growth|"
    r"rental yield|income generating|portfolio"
)

customer_profile["persona_investment_focused"] = (
    primary_intent.str.contains(
        investment_keywords,
        regex=True,
        na=False
    )
    | finance_type.str.contains(
        investment_keywords,
        regex=True,
        na=False
    )
    | buyer_position.str.contains(
        investment_keywords,
        regex=True,
        na=False
    )
)

# 4. Early-Stage Browsers / Window Shoppers
customer_profile["persona_early_stage_browser"] = (
    (
        (customer_profile["total_sessions"] > 0)
        | (customer_profile["total_property_views"] > 0)
    )
    & (customer_profile["total_leads"] == 0)
)

# 5. Slow-Progress Leads
customer_profile["persona_slow_progress_lead"] = (
    (customer_profile["open_or_unconverted_leads"] > 0)
    & (
        customer_profile["days_since_observed_activity"] >= 30
    )
    & (customer_profile["completed_transactions"] == 0)
)

# 6. Rental-Ready Customers
rental_intent_keywords = (
    r"rent|rental|tenant|letting|lease|accommodation"
)

rental_intent = primary_intent.str.contains(
    rental_intent_keywords,
    regex=True,
    na=False
)

rental_listing_interest = listing_purpose.str.contains(
    r"rent|rental|letting",
    regex=True,
    na=False
)

rent_budget = pd.to_numeric(
    customer_profile.get(
        "stated_rent_budget_pcm_gbp",
        pd.Series(0, index=customer_profile.index)
    ),
    errors="coerce"
).fillna(0)

customer_profile["persona_rental_ready"] = (
    rental_intent
    | (
        rental_listing_interest
        & (rent_budget > 0)
    )
)

# 7. Overseas Property Buyers
customer_profile["persona_overseas_property_buyer"] = (
    overseas_flag
    & (
        (customer_profile["total_property_views"] > 0)
        | (customer_profile["total_leads"] > 0)
        | customer_profile["persona_investment_focused"]
    )
)

# 8. Highly Engaged Property Explorers
customer_profile["persona_highly_engaged_explorer"] = (
    (
        customer_profile["total_property_views"] >= views_threshold
    )
    | (
        (customer_profile["total_sessions"] >= sessions_threshold)
        & (
            customer_profile["unique_properties_viewed"]
            >= unique_properties_threshold
        )
    )
    | (customer_profile["total_site_visits"] >= 2)
) & (
    customer_profile["total_property_views"] > 0
)

print("Eight behavioural personas created.")

75th percentile property views: 138.0
75th percentile sessions: 0.23052531537278517
75th percentile unique properties: 111.0
Eight behavioural personas created.


#### The sessions threshold of 0.2305 looks unusually low for a session count. it should inspect the actual values before using it to classify highly engaged customers.

In [33]:
# Inspect actual customer activity distributions

activity_columns = [
    "total_sessions",
    "total_property_views",
    "unique_properties_viewed",
    "total_site_visits"
]

for column in activity_columns:

    if column in customer_profile.columns:

        print(f"\n{column.upper()}")

        print(
            customer_profile[column]
            .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
        )

        print(
            "Unique values:",
            customer_profile[column].nunique()
        )

        print(
            "Missing values:",
            customer_profile[column].isna().sum()
        )


TOTAL_SESSIONS
count    1.020000e+05
mean     3.120815e-17
std      1.000005e+00
min     -7.955494e-01
25%     -5.795337e-01
50%     -3.095140e-01
75%      2.305253e-01
90%      1.040584e+00
95%      1.850643e+00
max      2.285817e+01
Name: total_sessions, dtype: float64
Unique values: 219
Missing values: 0

TOTAL_PROPERTY_VIEWS
count    102000.000000
mean        125.830745
std         207.282074
min           0.000000
25%          21.000000
50%          60.000000
75%         138.000000
90%         295.000000
95%         497.000000
max        5029.000000
Name: total_property_views, dtype: float64
Unique values: 1630
Missing values: 0

UNIQUE_PROPERTIES_VIEWED
count    102000.000000
mean         94.449794
std         132.813757
min           0.000000
25%          20.000000
50%          53.000000
75%         111.000000
90%         217.000000
95%         357.000000
max        2049.000000
Name: unique_properties_viewed, dtype: float64
Unique values: 1074
Missing values: 0

TOTAL_SITE_VISI

In [ ]:
#####=======> Check whether the eight personas were populated
# Count customers in each behavioural persona

persona_mapping = {
    "Repeat Buyers": "persona_repeat_buyer",
    "High-Intent Buyers": "persona_high_intent_buyer",
    "Investment-Focused Customers": "persona_investment_focused",
    "Early-Stage Browsers": "persona_early_stage_browser",
    "Slow-Progress Leads": "persona_slow_progress_lead",
    "Rental-Ready Customers": "persona_rental_ready",
    "Overseas Property Buyers": "persona_overseas_property_buyer",
    "Highly Engaged Property Explorers": "persona_highly_engaged_explorer"
}

persona_results = []

for persona, column in persona_mapping.items():

    count = int(customer_profile[column].sum())

    percentage = (
        count / len(customer_profile) * 100
    )

    persona_results.append({
        "Persona": persona,
        "Customer Count": count,
        "Customer Percentage": round(percentage, 2)
    })

persona_results_df = pd.DataFrame(persona_results)

display(
    persona_results_df.sort_values(
        "Customer Count",
        ascending=False
    )
)

,Persona,Customer Count,Customer Percentage
7,Highly Engaged Property Explorers,61351,60.15
1,High-Intent Buyers,53597,52.55
4,Slow-Progress Leads,46045,45.14
3,Early-Stage Browsers,39119,38.35
5,Rental-Ready Customers,5859,5.74
0,Repeat Buyers,1535,1.50
2,Investment-Focused Customers,0,0.00
6,Overseas Property Buyers,0,0.00


In [35]:
######=======> check overlapping persona
persona_columns = list(persona_mapping.values())

customer_profile["persona_count"] = (
    customer_profile[persona_columns].sum(axis=1)
)

print(
    "Total customers:",
    customer_profile["customer_id"].nunique()
)

print(
    "Customers with no persona:",
    (customer_profile["persona_count"] == 0).sum()
)

print(
    "Customers with one persona:",
    (customer_profile["persona_count"] == 1).sum()
)

print(
    "Customers with multiple personas:",
    (customer_profile["persona_count"] > 1).sum()
)

display(
    customer_profile[
        ["customer_id", "persona_count"] + persona_columns
    ].head(10)
)

Total customers: 102000
Customers with no persona: 1851
Customers with one persona: 27747
Customers with multiple personas: 72402


,customer_id,persona_count,persona_repeat_buyer,persona_high_intent_buyer,persona_investment_focused,persona_early_stage_browser,persona_slow_progress_lead,persona_rental_ready,persona_overseas_property_buyer,persona_highly_engaged_explorer
0,VLX-C00127559,3,False,False,False,True,False,True,False,True
1,VLX-C00168848,1,False,False,False,True,False,False,False,False
2,VLX-C00155207,2,False,True,False,False,True,False,False,False
3,VLX-C00247948,1,False,False,False,True,False,False,False,False
4,VLX-C00238325,1,False,False,False,True,False,False,False,False
5,VLX-C00097220,1,False,False,False,True,False,False,False,False
6,VLX-C00259256,2,True,False,False,False,False,False,False,True
7,VLX-C00297478,1,False,False,False,True,False,False,False,False
8,VLX-C00175053,1,False,False,False,True,False,False,False,False
9,VLX-C00131607,3,False,True,False,False,True,False,False,True


In [36]:
########=========> Inspect session threshold before segment summary
# Examine the actual session values

display(
    customer_profile[
        ["customer_id", "total_sessions"]
    ]
    .sort_values(
        "total_sessions",
        ascending=False
    )
    .head(20)
)

print(
    "Session data type:",
    customer_profile["total_sessions"].dtype
)

print(
    "Minimum sessions:",
    customer_profile["total_sessions"].min()
)

print(
    "Maximum sessions:",
    customer_profile["total_sessions"].max()
)

,customer_id,total_sessions
33017,VLX-C00122006,22.858173
10277,VLX-C00055871,20.049968
82279,VLX-C00126492,16.485709
90801,VLX-C00211516,14.649575
15883,VLX-C00294576,14.595571
38741,VLX-C00297652,14.001528
25691,VLX-C00174610,13.947524
24503,VLX-C00159748,13.191469
92507,VLX-C00062432,12.651430
80167,VLX-C00283614,12.381410


Session data type: float64
Minimum sessions: -0.7955493994156476
Maximum sessions: 22.858172973075593


In [ ]:
######=======> Recover actual session counts
# Calculate actual session counts from raw web_sessions

session_features = con.sql("""
    SELECT
        customer_id,
        COUNT(DISTINCT session_id) AS actual_total_sessions,
        COUNT(DISTINCT CAST(session_start AS DATE)) AS active_session_days,
        MAX(session_start) AS last_session_date
    FROM web_sessions
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
""").df()

print("Session features shape:", session_features.shape)

display(
    session_features.sort_values(
        "actual_total_sessions",
        ascending=False
    ).head(10)
)

print("\nActual session distribution:")

display(
    session_features["actual_total_sessions"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Session features shape: (96368, 4)


,customer_id,actual_total_sessions,active_session_days,last_session_date
68530,VLX-C00122006,438,197,2024-12-24 21:02:58
61860,VLX-C00055871,386,249,2024-06-23 22:32:44
11531,VLX-C00126492,320,184,2026-09-29 21:37:01
79577,VLX-C00211516,286,86,2026-08-15 17:49:27
15121,VLX-C00294576,285,220,2024-12-22 12:06:38
6278,VLX-C00297652,274,190,2025-07-10 19:40:19
81630,VLX-C00174610,273,204,2025-04-15 11:59:43
34022,VLX-C00159748,259,71,2024-03-18 23:48:33
78071,VLX-C00062432,249,107,2026-09-30 15:03:35
64350,VLX-C00283614,244,151,2026-09-26 22:12:33



Actual session distribution:


count    96368.000000
mean        15.592261
std         18.695037
min          1.000000
25%          4.000000
50%         10.000000
75%         19.000000
90%         35.000000
95%         50.000000
max        438.000000
Name: actual_total_sessions, dtype: float64

In [39]:
# ==========================================
# CORRECT SESSION FEATURES IN CUSTOMER PROFILE
# ==========================================

# 1. Check existing session-related columns

print("Existing session columns:")

print([
    col for col in customer_profile.columns
    if "session" in col.lower()
])

# 2. Remove old session features to prevent duplicate columns

columns_to_remove = [
    "total_sessions",
    "actual_total_sessions",
    "active_session_days",
    "last_session_date",
    "total_sessions_x",
    "total_sessions_y",
    "active_session_days_x",
    "active_session_days_y",
    "last_session_date_x",
    "last_session_date_y"
]

customer_profile = customer_profile.drop(
    columns=columns_to_remove,
    errors="ignore"
)

# 3. Merge the actual session features

customer_profile = customer_profile.merge(
    session_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# 4. Replace missing activity values with zero

activity_columns = [
    "actual_total_sessions",
    "active_session_days"
]

for column in activity_columns:

    customer_profile[column] = (
        customer_profile[column].fillna(0)
    )

# 5. Check the corrected results

print("Customer profile shape:", customer_profile.shape)

print("\nSession feature columns:")

display(
    customer_profile[
        [
            "customer_id",
            "actual_total_sessions",
            "active_session_days",
            "last_session_date"
        ]
    ].head(10)
)

print("\nActual session distribution:")

display(
    customer_profile["actual_total_sessions"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

Existing session columns:
['total_sessions_x', 'avg_session_duration', 'total_session_duration', 'night_session_share_x', 'total_sessions_y', 'active_session_days_x', 'avg_session_duration_sec', 'total_session_duration_sec', 'latest_session_date', 'actual_total_sessions', 'active_session_days_y', 'last_session_date']
Customer profile shape: (102000, 64)

Session feature columns:


,customer_id,actual_total_sessions,active_session_days,last_session_date
0,VLX-C00127559,58.0,39.0,2023-03-17 21:04:18
1,VLX-C00168848,6.0,5.0,2023-04-11 21:53:57
2,VLX-C00155207,4.0,4.0,2023-06-11 22:27:18
3,VLX-C00247948,5.0,5.0,2024-04-22 03:47:09
4,VLX-C00238325,2.0,2.0,2023-01-05 15:19:59
5,VLX-C00097220,1.0,1.0,2023-03-25 16:10:37
6,VLX-C00259256,5.0,5.0,2024-02-11 22:53:06
7,VLX-C00297478,5.0,5.0,2023-03-29 07:11:31
8,VLX-C00175053,8.0,7.0,2023-02-02 19:49:34
9,VLX-C00131607,3.0,3.0,2023-09-14 05:55:29



Actual session distribution:


count    102000.000000
mean         14.731324
std          18.517261
min           0.000000
25%           4.000000
50%           9.000000
75%          19.000000
90%          34.000000
95%          49.000000
max         438.000000
Name: actual_total_sessions, dtype: float64

In [42]:

# ==========================================
# CORRECT SESSION FEATURES IN CUSTOMER_PROFILE
# ==========================================

# 1. Remove previous session count columns
# Keep unrelated features such as session duration and night activity.

session_columns_to_remove = [
    "total_sessions",
    "total_sessions_x",
    "total_sessions_y",
    "actual_total_sessions",
    "active_session_days",
    "active_session_days_x",
    "active_session_days_y",
    "last_session_date",
    "last_session_date_x",
    "last_session_date_y"
]

customer_profile = customer_profile.drop(
    columns=session_columns_to_remove,
    errors="ignore"
)

# 2. Prepare actual session features
session_features_clean = session_features[
    [
        "customer_id",
        "actual_total_sessions",
        "active_session_days",
        "last_session_date"
    ]
].copy()

# 3. Merge actual session counts
customer_profile = customer_profile.merge(
    session_features_clean,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# 4. Replace missing session counts with zero
customer_profile["actual_total_sessions"] = (
    customer_profile["actual_total_sessions"].fillna(0)
)

customer_profile["active_session_days"] = (
    customer_profile["active_session_days"].fillna(0)
)

# 5. Validate customer records
print("Customer records:", len(customer_profile))
print("Unique customers:", customer_profile["customer_id"].nunique())

print("\nSession columns:")
print([
    col for col in customer_profile.columns
    if "session" in col.lower()
])

# 6. Display actual session distribution
display(
    customer_profile["actual_total_sessions"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)


Customer records: 102000
Unique customers: 102000

Session columns:
['avg_session_duration', 'total_session_duration', 'night_session_share_x', 'avg_session_duration_sec', 'total_session_duration_sec', 'latest_session_date', 'actual_total_sessions_x', 'actual_total_sessions_y', 'actual_total_sessions', 'active_session_days', 'last_session_date']


count    102000.000000
mean         14.731324
std          18.517261
min           0.000000
25%           4.000000
50%           9.000000
75%          19.000000
90%          34.000000
95%          49.000000
max         438.000000
Name: actual_total_sessions, dtype: float64

In [43]:
# ==========================================
# CUSTOMER ENGAGEMENT THRESHOLDS
# ==========================================

engagement_features = [
    "actual_total_sessions",
    "total_property_views",
    "unique_properties_viewed"
]

engagement_thresholds = {}

for column in engagement_features:

    threshold = customer_profile[column].quantile(0.75)

    engagement_thresholds[column] = threshold

    print(f"{column}: {threshold:.2f}")

print("\nEngagement thresholds calculated successfully.")

actual_total_sessions: 19.00
total_property_views: 138.00
unique_properties_viewed: 111.00

Engagement thresholds calculated successfully.


In [ ]:
# Investment-related customer characteristics

investment_columns = [
    "primary_intent",
    "buyer_position",
    "finance_type",
    "customer_type"
]

for column in investment_columns:

    if column in customer_profile.columns:

        print(f"\n{column.upper()}")

        display(
            customer_profile[column]
            .value_counts(dropna=False)
            .head(20)
        )

'customer_type'

In [45]:
# Overseas customer distribution

for column in [
    "is_overseas",
    "country_of_residence",
    "uk_region_of_residence"
]:

    if column in customer_profile.columns:

        print(f"\n{column.upper()}")

        display(
            customer_profile[column]
            .value_counts(dropna=False)
            .head(20)
        )

In [47]:
# Inspect actual transaction types

display(
    con.sql("""
        SELECT
            transaction_type,
            COUNT(*) AS transaction_count,
            COUNT(DISTINCT customer_id) AS customer_count
        FROM transactions
        GROUP BY transaction_type
        ORDER BY transaction_count DESC
    """).df()
)

,transaction_type,transaction_count,customer_count
0,Resale purchase,9063,8359
1,Assured shorthold tenancy,3327,2992
2,New-build purchase,1866,1830
3,Company let,1311,1126
4,Off-plan purchase,1098,1083


In [48]:
######=======> inpect property listing purposes
display(
    con.sql("""
        SELECT
            listing_purpose,
            COUNT(*) AS property_count
        FROM properties
        GROUP BY listing_purpose
        ORDER BY property_count DESC
    """).df()
)

,listing_purpose,property_count
0,Sale,12550
1,Rent,4117


In [49]:
# ==========================================
# INVESTMENT PERSONA: TRANSACTION + PROPERTY EVIDENCE
# ==========================================

# 1. Aggregate actual lead, transaction and property information

investment_evidence = con.sql("""
    WITH lead_summary AS (
        SELECT
            customer_id,
            STRING_AGG(DISTINCT CAST(intent AS VARCHAR), ' | ') AS lead_intents
        FROM leads
        WHERE customer_id IS NOT NULL
        GROUP BY customer_id
    ),

    transaction_summary AS (
        SELECT
            t.customer_id,

            COUNT(DISTINCT t.transaction_id) AS total_transactions,

            STRING_AGG(
                DISTINCT CAST(t.transaction_type AS VARCHAR),
                ' | '
            ) AS transaction_types,

            STRING_AGG(
                DISTINCT CAST(p.listing_purpose AS VARCHAR),
                ' | '
            ) AS transacted_property_purposes,

            STRING_AGG(
                DISTINCT CAST(p.property_type AS VARCHAR),
                ' | '
            ) AS transacted_property_types

        FROM transactions t
        LEFT JOIN properties p
            ON t.property_id = p.property_id

        WHERE t.customer_id IS NOT NULL
        GROUP BY t.customer_id
    )

    SELECT
        COALESCE(l.customer_id, t.customer_id) AS customer_id,
        l.lead_intents,
        t.total_transactions,
        t.transaction_types,
        t.transacted_property_purposes,
        t.transacted_property_types

    FROM lead_summary l
    FULL OUTER JOIN transaction_summary t
        ON l.customer_id = t.customer_id
""").df()


# 2. Remove previous versions of these evidence columns

customer_profile = customer_profile.drop(
    columns=[
        "lead_intents",
        "total_transactions",
        "transaction_types",
        "transacted_property_purposes",
        "transacted_property_types"
    ],
    errors="ignore"
)

# 3. Merge evidence into the customer profile

customer_profile = customer_profile.merge(
    investment_evidence,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# 4. Prepare text fields for investment classification

investment_text_columns = [
    "primary_intent",
    "buyer_position",
    "finance_type",
    "customer_type",
    "lead_intents",
    "transaction_types",
    "transacted_property_purposes",
    "transacted_property_types"
]

for column in investment_text_columns:
    if column not in customer_profile.columns:
        customer_profile[column] = ""

    customer_profile[column] = (
        customer_profile[column]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[_-]+", " ", regex=True)
    )

# 5. Identify explicit investment signals

investment_pattern = (
    r"\binvest(?:ment|or|ing)?\b"
    r"|buy to let"
    r"|buy to let mortgage"
    r"|\blandlord\b"
    r"|rental income"
    r"|rental yield"
    r"|capital growth"
    r"|property portfolio"
    r"|investment property"
)

customer_profile["investment_intent_signal"] = (
    customer_profile[
        [
            "primary_intent",
            "buyer_position",
            "finance_type",
            "customer_type",
            "lead_intents"
        ]
    ]
    .apply(
        lambda row: row.str.contains(
            investment_pattern,
            regex=True,
            na=False
        ).any(),
        axis=1
    )
)

# 6. Identify explicit investment transaction signals

transaction_pattern = (
    r"\binvest(?:ment|or)?\b"
    r"|buy to let"
    r"|rental investment"
    r"|\blandlord\b"
)

customer_profile["investment_transaction_signal"] = (
    customer_profile["transaction_types"]
    .str.contains(transaction_pattern, regex=True, na=False)
)

# 7. Identify rental property evidence linked to transactions

customer_profile["rental_property_transaction_signal"] = (
    customer_profile["transacted_property_purposes"]
    .str.contains(
        r"\brent(?:al)?\b|\blet(?:ting)?\b",
        regex=True,
        na=False
    )
)

# 8. Final investment-focused persona
# Explicit investment intent or investment transaction evidence.
# Rental property evidence is retained as supporting information,
# but rental interest alone does not make someone an investor.

customer_profile["persona_investment_focused"] = (
    customer_profile["investment_intent_signal"]
    | customer_profile["investment_transaction_signal"]
)

# 9. Review the classification

print(
    "Investment-Focused Customers:",
    int(customer_profile["persona_investment_focused"].sum())
)

print(
    "Percentage:",
    round(
        customer_profile["persona_investment_focused"].mean() * 100,
        2
    ),
    "%"
)

display(
    customer_profile.loc[
        customer_profile["persona_investment_focused"],
        [
            "customer_id",
            "primary_intent",
            "buyer_position",
            "finance_type",
            "lead_intents",
            "transaction_types",
            "transacted_property_purposes",
            "persona_investment_focused"
        ]
    ].head(20)
)

Investment-Focused Customers: 0
Percentage: 0.0 %


,customer_id,primary_intent,buyer_position,finance_type,lead_intents,transaction_types,transacted_property_purposes,persona_investment_focused


In [58]:
# ==========================================
# RETRIEVE OVERSEAS CUSTOMER INFORMATION
# ==========================================

overseas_features = con.sql("""
    SELECT
        customer_id,
        is_overseas,
        country_of_residence
    FROM customers
""").df()

print("Overseas features shape:", overseas_features.shape)

display(overseas_features.head(10))

print("\nOverseas indicator values:")
display(
    overseas_features["is_overseas"].value_counts(dropna=False)
)

print("\nCountry of residence values:")
display(
    overseas_features["country_of_residence"]
    .value_counts(dropna=False)
    .head(20)
)

Overseas features shape: (102000, 3)


,customer_id,is_overseas,country_of_residence
0,VLX-C00127559,False,United Kingdom
1,VLX-C00168848,False,United Kingdom
2,VLX-C00155207,False,United Kingdom
3,VLX-C00247948,True,United States
4,VLX-C00238325,False,United Kingdom
5,VLX-C00097220,False,Great Britain
6,VLX-C00259256,False,United Kingdom
7,VLX-C00297478,False,UK
8,VLX-C00175053,False,United Kingdom
9,VLX-C00131607,True,Hong Kong



Overseas indicator values:


is_overseas
False    92404
True      9596
Name: count, dtype: int64


Country of residence values:


country_of_residence
United Kingdom          64406
UK                      14124
England                  7435
Great Britain            3664
U.K.                     2775
Hong Kong                1306
United Arab Emirates     1064
India                     863
Nigeria                   787
Singapore                 771
China                     711
United States             702
Canada                    545
UAE                       480
USA                       399
HK                        358
Malaysia                  327
Saudi Arabia              285
Qatar                     244
Hong Kong SAR             198
Name: count, dtype: int64

In [59]:
# ==========================================
# MERGE OVERSEAS CUSTOMER FEATURES
# ==========================================

customer_profile = customer_profile.drop(
    columns=[
        "is_overseas",
        "country_of_residence"
    ],
    errors="ignore"
)

customer_profile = customer_profile.merge(
    overseas_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("Customer profile shape:", customer_profile.shape)

print("\nOverseas columns:")
display(
    customer_profile[
        [
            "customer_id",
            "is_overseas",
            "country_of_residence"
        ]
    ].head(10)
)

Customer profile shape: (102000, 92)

Overseas columns:


,customer_id,is_overseas,country_of_residence
0,VLX-C00127559,False,United Kingdom
1,VLX-C00168848,False,United Kingdom
2,VLX-C00155207,False,United Kingdom
3,VLX-C00247948,True,United States
4,VLX-C00238325,False,United Kingdom
5,VLX-C00097220,False,Great Britain
6,VLX-C00259256,False,United Kingdom
7,VLX-C00297478,False,UK
8,VLX-C00175053,False,United Kingdom
9,VLX-C00131607,True,Hong Kong


In [60]:
# ==========================================
# OVERSEAS PROPERTY BUYER PERSONA
# ==========================================

# Normalise overseas indicator

overseas_values = (
    customer_profile["is_overseas"]
    .astype(str)
    .str.strip()
    .str.lower()
)

true_values = ["true", "1", "yes", "y", "overseas"]

customer_profile["overseas_indicator"] = (
    overseas_values.isin(true_values)
)

# Normalise country names

customer_profile["country_normalised"] = (
    customer_profile["country_of_residence"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

# UK country labels

uk_countries = [
    "uk",
    "u k",
    "gb",
    "great britain",
    "britain",
    "united kingdom",
    "england",
    "scotland",
    "wales",
    "northern ireland"
]

customer_profile["uk_resident"] = (
    customer_profile["country_normalised"].isin(uk_countries)
)

# Identify non-UK country evidence

customer_profile["overseas_country_signal"] = (
    customer_profile["country_normalised"].ne("")
    & ~customer_profile["uk_resident"]
)

# Final overseas classification

customer_profile["persona_overseas_property_buyer"] = (
    customer_profile["overseas_indicator"]
    | customer_profile["overseas_country_signal"]
)

# Results

print(
    "Overseas Property Buyers:",
    int(customer_profile["persona_overseas_property_buyer"].sum())
)

print(
    "Percentage:",
    round(
        customer_profile["persona_overseas_property_buyer"].mean() * 100,
        2
    ),
    "%"
)

display(
    customer_profile.loc[
        customer_profile["persona_overseas_property_buyer"],
        [
            "customer_id",
            "is_overseas",
            "country_of_residence",
            "overseas_indicator",
            "overseas_country_signal"
        ]
    ].head(20)
)

Overseas Property Buyers: 9596
Percentage: 9.41 %


,customer_id,is_overseas,country_of_residence,overseas_indicator,overseas_country_signal
3,VLX-C00247948,True,United States,True,True
9,VLX-C00131607,True,Hong Kong,True,True
16,VLX-C00246559,True,Hong Kong,True,True
17,VLX-C00198852,True,Hong Kong,True,True
18,VLX-C00151789,True,China,True,True
25,VLX-C00032460,True,Nigeria,True,True
39,VLX-C00241750,True,United States,True,True
62,VLX-C00124051,True,Singapore,True,True
75,VLX-C00144485,True,United Arab Emirates,True,True
79,VLX-C00281272,True,Singapore,True,True


In [62]:
# ==========================================
# PERSONA VALIDATION
# ==========================================

print("INVESTMENT PERSONA")
print(
    customer_profile["persona_investment_focused"]
    .value_counts(dropna=False)
)

print("\nOVERSEAS PERSONA")
print(
    customer_profile["persona_overseas_property_buyer"]
    .value_counts(dropna=False)
)

print("\nOVERSEAS COUNTRY DISTRIBUTION")

display(
    customer_profile.loc[
        customer_profile["persona_overseas_property_buyer"],
        "country_of_residence"
    ].value_counts(dropna=False).head(20)
)

print("\nINVESTMENT EVIDENCE")

display(
    customer_profile.loc[
        customer_profile["persona_investment_focused"],
        [
            "investment_intent_signal",
            "investment_transaction_signal",
            "rental_property_transaction_signal"
        ]
    ].sum().to_frame("Customer count")
)

INVESTMENT PERSONA
persona_investment_focused
False    102000
Name: count, dtype: int64

OVERSEAS PERSONA
persona_overseas_property_buyer
False    92404
True      9596
Name: count, dtype: int64

OVERSEAS COUNTRY DISTRIBUTION


country_of_residence
Hong Kong               1306
United Arab Emirates    1064
India                    863
Nigeria                  787
Singapore                771
China                    711
United States            702
Canada                   545
UAE                      480
USA                      399
HK                       358
Malaysia                 327
Saudi Arabia             285
Qatar                    244
Hong Kong SAR            198
US                       129
Mainland China           100
U.A.E.                    94
PRC                       92
SG                        88
Name: count, dtype: int64


INVESTMENT EVIDENCE


,Customer count
investment_intent_signal,0
investment_transaction_signal,0
rental_property_transaction_signal,0


In [63]:
#######=======> Update the eight-persona mapping
persona_mapping = {
    "Repeat Buyers": "persona_repeat_buyer",
    "High-Intent Buyers": "persona_high_intent_buyer",
    "Investment-Focused Customers": "persona_investment_focused",
    "Early-Stage Browsers": "persona_early_stage_browser",
    "Slow-Progress Leads": "persona_slow_progress_lead",
    "Rental-Ready Customers": "persona_rental_ready",
    "Overseas Property Buyers": "persona_overseas_property_buyer",
    "Highly Engaged Property Explorers": "persona_highly_engaged_explorer"
}

persona_summary = pd.DataFrame([
    {
        "Persona": persona,
        "Customer Count": int(customer_profile[column].sum()),
        "Customer Percentage": round(
            customer_profile[column].mean() * 100, 2
        )
    }
    for persona, column in persona_mapping.items()
])

display(
    persona_summary.sort_values(
        "Customer Count",
        ascending=False
    ).reset_index(drop=True)
)

,Persona,Customer Count,Customer Percentage
0,Highly Engaged Property Explorers,61351,60.15
1,High-Intent Buyers,53597,52.55
2,Slow-Progress Leads,46045,45.14
3,Early-Stage Browsers,39119,38.35
4,Overseas Property Buyers,9596,9.41
5,Rental-Ready Customers,5859,5.74
6,Repeat Buyers,1535,1.50
7,Investment-Focused Customers,0,0.00


In [64]:
# ==========================================
# CUSTOMER PROPERTY AFFORDABILITY FEATURES
# ==========================================

affordability_features = con.sql("""
    SELECT
        ws.customer_id,

        -- Purchase property affordability
        COUNT(DISTINCT CASE
            WHEN p.asking_price_gbp > 0
            THEN pv.property_id
        END) AS purchase_properties_viewed,

        MIN(CASE
            WHEN p.asking_price_gbp > 0
            THEN p.asking_price_gbp
        END) AS cheapest_viewed_purchase_price,

        MEDIAN(CASE
            WHEN p.asking_price_gbp > 0
            THEN p.asking_price_gbp
        END) AS median_viewed_purchase_price,

        -- Rental property affordability
        COUNT(DISTINCT CASE
            WHEN p.rent_pcm_gbp > 0
            THEN pv.property_id
        END) AS rental_properties_viewed,

        MIN(CASE
            WHEN p.rent_pcm_gbp > 0
            THEN p.rent_pcm_gbp
        END) AS cheapest_viewed_rent_pcm,

        MEDIAN(CASE
            WHEN p.rent_pcm_gbp > 0
            THEN p.rent_pcm_gbp
        END) AS median_viewed_rent_pcm

    FROM web_sessions ws

    INNER JOIN property_views pv
        ON ws.session_id = pv.session_id

    INNER JOIN properties p
        ON pv.property_id = p.property_id

    WHERE ws.customer_id IS NOT NULL

    GROUP BY ws.customer_id
""").df()

print("Affordability features:", affordability_features.shape)

display(affordability_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Affordability features: (96096, 7)


,customer_id,purchase_properties_viewed,cheapest_viewed_purchase_price,median_viewed_purchase_price,rental_properties_viewed,cheapest_viewed_rent_pcm,median_viewed_rent_pcm
0,VLX-C00058910,878,75000,210000.0,0,<NA>,NaN
1,VLX-C00090102,122,95000,200000.0,0,<NA>,NaN
2,VLX-C00080411,96,95000,170000.0,0,<NA>,NaN
3,VLX-C00214242,757,120000,440000.0,0,<NA>,NaN
4,VLX-C00148840,29,85000,190000.0,0,<NA>,NaN


In [65]:
# ==========================================
# MERGE CUSTOMER AFFORDABILITY FEATURES
# ==========================================

# Remove old versions if this cell is rerun

affordability_columns = [
    "purchase_properties_viewed",
    "cheapest_viewed_purchase_price",
    "median_viewed_purchase_price",
    "rental_properties_viewed",
    "cheapest_viewed_rent_pcm",
    "median_viewed_rent_pcm"
]

customer_profile = customer_profile.drop(
    columns=affordability_columns,
    errors="ignore"
)

# Merge features

customer_profile = customer_profile.merge(
    affordability_features,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Replace missing property counts with zero

for column in [
    "purchase_properties_viewed",
    "rental_properties_viewed"
]:
    customer_profile[column] = (
        customer_profile[column].fillna(0)
    )

# Convert budget and property price fields to numeric

budget_columns = [
    "stated_budget_gbp",
    "stated_rent_budget_pcm_gbp",
    "cheapest_viewed_purchase_price",
    "median_viewed_purchase_price",
    "cheapest_viewed_rent_pcm",
    "median_viewed_rent_pcm"
]

for column in budget_columns:
    if column in customer_profile.columns:
        customer_profile[column] = pd.to_numeric(
            customer_profile[column],
            errors="coerce"
        )

print("Customer records:", len(customer_profile))

Customer records: 102000


In [66]:
# ==========================================
# PURCHASE AFFORDABILITY RENTAL AFFORDABILITY
# ==========================================

customer_profile["purchase_affordability_ratio"] = (
    customer_profile["cheapest_viewed_purchase_price"]
    / customer_profile["stated_budget_gbp"].replace(0, np.nan)
)

# Within budget: at least one viewed property is affordable

customer_profile["purchase_within_budget"] = (
    customer_profile["stated_budget_gbp"].gt(0)
    & customer_profile["cheapest_viewed_purchase_price"].notna()
    & (
        customer_profile["cheapest_viewed_purchase_price"]
        <= customer_profile["stated_budget_gbp"]
    )
)

# ==========================================
# RENTAL AFFORDABILITY
# ==========================================

customer_profile["rental_affordability_ratio"] = (
    customer_profile["cheapest_viewed_rent_pcm"]
    / customer_profile["stated_rent_budget_pcm_gbp"].replace(0, np.nan)
)

customer_profile["rental_within_budget"] = (
    customer_profile["stated_rent_budget_pcm_gbp"].gt(0)
    & customer_profile["cheapest_viewed_rent_pcm"].notna()
    & (
        customer_profile["cheapest_viewed_rent_pcm"]
        <= customer_profile["stated_rent_budget_pcm_gbp"]
    )
)

In [67]:
# ==========================================
# PURCHASE AFFORDABILITY STATUS
# ==========================================

def classify_affordability(budget, cheapest_price, properties_viewed):

    if pd.isna(budget) or budget <= 0:
        return "Budget Not Provided"

    if properties_viewed == 0 or pd.isna(cheapest_price):
        return "No Relevant Property Views"

    if cheapest_price <= budget:
        return "Within Budget"

    elif cheapest_price <= budget * 1.10:
        return "Near Budget"

    else:
        return "Above Budget"


customer_profile["purchase_affordability_status"] = customer_profile.apply(
    lambda row: classify_affordability(
        row["stated_budget_gbp"],
        row["cheapest_viewed_purchase_price"],
        row["purchase_properties_viewed"]
    ),
    axis=1
)

# ==========================================
# RENTAL AFFORDABILITY STATUS
# ==========================================

customer_profile["rental_affordability_status"] = customer_profile.apply(
    lambda row: classify_affordability(
        row["stated_rent_budget_pcm_gbp"],
        row["cheapest_viewed_rent_pcm"],
        row["rental_properties_viewed"]
    ),
    axis=1
)

display(
    customer_profile[
        [
            "customer_id",
            "stated_budget_gbp",
            "purchase_affordability_status",
            "stated_rent_budget_pcm_gbp",
            "rental_affordability_status"
        ]
    ].head(10)
)

,customer_id,stated_budget_gbp,purchase_affordability_status,stated_rent_budget_pcm_gbp,rental_affordability_status
0,VLX-C00127559,-0.167595,Budget Not Provided,2.060513,Above Budget
1,VLX-C00168848,-0.625589,Budget Not Provided,-0.080618,Budget Not Provided
2,VLX-C00155207,3.063808,Above Budget,-0.080618,Budget Not Provided
3,VLX-C00247948,-0.803698,Budget Not Provided,-0.080618,Budget Not Provided
4,VLX-C00238325,-0.167595,Budget Not Provided,-0.452989,Budget Not Provided
5,VLX-C00097220,1.155499,Above Budget,-0.080618,Budget Not Provided
6,VLX-C00259256,-0.676477,Budget Not Provided,-0.080618,Budget Not Provided
7,VLX-C00297478,3.318249,Above Budget,-0.080618,Budget Not Provided
8,VLX-C00175053,-0.167595,Budget Not Provided,-0.080618,Budget Not Provided
9,VLX-C00131607,-0.167595,Budget Not Provided,-0.080618,Budget Not Provided


In [68]:
# ==========================================
# AFFORDABILITY-MATCHED CUSTOMER PERSONA
# ==========================================

customer_profile["persona_affordability_matched"] = (
    customer_profile["purchase_within_budget"]
    | customer_profile["rental_within_budget"]
)

print(
    "Affordability-Matched Customers:",
    int(customer_profile["persona_affordability_matched"].sum())
)

print(
    "Percentage:",
    round(
        customer_profile["persona_affordability_matched"].mean() * 100,
        2
    ),
    "%"
)

Affordability-Matched Customers: 0
Percentage: 0.0 %


In [69]:
# ==========================================
# PURCHASE AFFORDABILITY SUMMARY
# ==========================================

purchase_summary = (
    customer_profile["purchase_affordability_status"]
    .value_counts()
    .rename_axis("Affordability Status")
    .reset_index(name="Customer Count")
)

purchase_summary["Percentage"] = (
    purchase_summary["Customer Count"]
    / len(customer_profile) * 100
).round(2)

print("PURCHASE AFFORDABILITY")
display(purchase_summary)


# ==========================================
# RENTAL AFFORDABILITY SUMMARY
# ==========================================

rental_summary = (
    customer_profile["rental_affordability_status"]
    .value_counts()
    .rename_axis("Affordability Status")
    .reset_index(name="Customer Count")
)

rental_summary["Percentage"] = (
    rental_summary["Customer Count"]
    / len(customer_profile) * 100
).round(2)

print("RENTAL AFFORDABILITY")
display(rental_summary)

PURCHASE AFFORDABILITY


,Affordability Status,Customer Count,Percentage
0,Budget Not Provided,80270,78.7
1,Above Budget,20401,20.0
2,No Relevant Property Views,1329,1.3


RENTAL AFFORDABILITY


,Affordability Status,Customer Count,Percentage
0,Budget Not Provided,95560,93.69
1,Above Budget,5859,5.74
2,No Relevant Property Views,581,0.57


In [70]:
######========>Add the ninth persona to your mapping

persona_mapping["Affordability-Matched Customers"] = (
    "persona_affordability_matched"
)

persona_summary = pd.DataFrame([
    {
        "Persona": persona,
        "Customer Count": int(customer_profile[column].sum()),
        "Customer Percentage": round(
            customer_profile[column].mean() * 100, 2
        )
    }
    for persona, column in persona_mapping.items()
])

display(
    persona_summary.sort_values(
        "Customer Count",
        ascending=False
    ).reset_index(drop=True)
)

,Persona,Customer Count,Customer Percentage
0,Highly Engaged Property Explorers,61351,60.15
1,High-Intent Buyers,53597,52.55
2,Slow-Progress Leads,46045,45.14
3,Early-Stage Browsers,39119,38.35
4,Overseas Property Buyers,9596,9.41
5,Rental-Ready Customers,5859,5.74
6,Repeat Buyers,1535,1.50
7,Investment-Focused Customers,0,0.00
8,Affordability-Matched Customers,0,0.00
